# 03b · MNIST: protocolo de cierre

Mismo protocolo que el Swiss roll (notebook 05) y el toro (02b): DMAP congelado, `vae` con kl = 10⁻³ y 10 000 pasos, desde cero y por continuación en m, réplicas independientes. **Cambio de v5:** en MNIST la submuestra de imágenes ahora sigue a `data_seed`, así que cada semilla es una réplica con otras imágenes de entrenamiento (antes solo cambiaba la inicialización). El test es un subconjunto del test oficial.

**Lo que MNIST puede y no puede decir.** No hay carta conocida: no hay oráculo de carta, ni piso, ni métrica de despliegue. Aquí se mide (i) la información del prefijo sobre los datos (lectura afín de x desde ψ₁…ψ_m, sin entrenar) y (ii) si la continuación espectral mejora la reconstrucción en datos reales. La exactitud kNN del dígito a partir del latente es una **sonda de estructura de clases**, no una carta. No se afirma que MNIST sea una variedad suave de dimensión 2.

| | Pregunta | Diseño | Criterio (fijado antes de correr) |
|---|---|---|---|
| **M1** | ¿Cuánta información sobre x tiene cada prefijo? | lectura afín de x y `direct` (decoder sin cuello de botella) | se reporta la curva; sin predicción teórica |
| **M2** | ¿La continuación mejora sobre entrenar desde cero? | latente 2, 8 réplicas, mismas semillas en ambos | Wilcoxon pareado (misma réplica) unilateral en m = M: p < 0.05 |
| **M3** | ¿Importa el orden espectral? | igual que M2 con orden aleatorio | en m = M (mismos modos): Mann-Whitney unilateral p < 0.05 para `vae_warm`; el control `vae` desde cero no debe diferir |
| **M4** | Comparador convencional | `plainvae` (MLP sobre los píxeles), latente 2 | MSE_x frente a `vae_warm` en la m elegida por validación; Mann-Whitney |

β sin etiquetas en dos etapas: (1) barrido barato con controles espectrales y la lectura afín de x (validación); (2) confirmación con el decoder `direct` en 4 candidatos repartidos en toda la rejilla admisible (la lectura afín favorece núcleos anchos, así que no se limita a sus vecinos), criterio: MSE de **validación** de x. Nunca se usan las etiquetas ni el test para elegir β.

Duración estimada ~3.5 h en dos T4. Necesita Internet si `mnist.npz` no está en `/kaggle/input`.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_03b.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_03b.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β sin etiquetas (mismo n que el experimento, ~30 min)

Las columnas `knn_label_acc_*` usan las etiquetas y son solo diagnóstico.

In [ ]:
import time
import pandas as pd
from IPython.display import display, Image
import protocol as P

N = 3000
from data import ensure_mnist, valid_mnist
# Se descarga UNA vez aquí, antes de lanzar los dos workers (antes ambos descargaban a la vez al mismo
# archivo y uno leía un ZIP a medio escribir). ensure_mnist valida y reemplaza un archivo corrupto.
found = [p for p in sorted(Path('/kaggle/input').rglob('mnist.npz')) if valid_mnist(p)]
MNIST = str(found[0]) if found else str(ensure_mnist('/kaggle/working/mnist.npz'))
print('MNIST:', MNIST)

cfg_sweep = json.loads((ROOT / 'config_beta_sweep_mnist.json').read_text())
cfg_sweep.update(n=N, data=MNIST)
sel = P.run_sweep(cfg_sweep, ROOT, 'beta_sweep_mnist')
assert sel['selected_factor'] is not None, 'Ningún factor pasó los controles espectrales: amplía la lista.'
SWEEP = Path(cfg_sweep['out'])
sw = pd.read_csv(SWEEP / 'beta_sweep.csv')
SEL_MODES = cfg_sweep['selection_modes']
cols = [c for c in ['kernel_length', 'ok', 'perron_gap', 'min_eigenvalue', 'nystrom_training_relative_error',
                    'ambient_val_selection'] + [f'ambient_val_m{m}' for m in SEL_MODES]
        + [f'knn_label_acc_m{m}' for m in SEL_MODES] if c in sw]
display(sw.groupby('factor')[cols].mean().style.format('{:.3g}').set_caption(
    f"etapa 1: seleccionado {sel['selected_factor']}, banda {sel['equivalent_band']}"))
display(Image(filename=str(SWEEP / 'beta_sweep.png')))

# ---- etapa 2: confirmación con el decoder direct (validación de x, sin etiquetas) ----
BASE = json.loads((ROOT / 'config_mnist.json').read_text())
BASE.update(n=N, alpha=1.0, data=MNIST)
COMMON = dict(P.COMMON)
if sel.get('edge_warning'):
    print('ATENCIÓN (etapa 1):', sel['edge_warning'])
# La lectura afín de la etapa 1 favorece núcleos anchos (autofunciones casi lineales), así que la
# etapa 2 NO se limita a sus vecinos: 4 candidatos repartidos en toda la rejilla admisible.
CAND = P.spread(sw, k=4)
CONF = [dict(tag=f'beta{f:g}', beta_factor=f, seeds=[0, 1], modes=SEL_MODES, models=['direct'],
             m_protocols=['scratch'], out=f'/kaggle/working/beta_confirm_mnist/f{f:g}') for f in CAND]
print('candidatos etapa 2 (repartidos en la rejilla):', CAND)
P.run_plan(CONF, BASE, ROOT, common=COMMON)
conf, FACTOR = P.confirm_direct({p['beta_factor']: p['out'] for p in CONF}, SEL_MODES)
display(conf.style.format('{:.4g}').set_caption('etapa 2: MSE de validación del decoder direct (media sobre semillas y m)'))
print(f"factor final = {FACTOR} (etapa 1: {sel['selected_factor']})")

## Plan y estimación de tiempo

El orden es la prioridad; `E_k8_spectral` (latente 8) es opcional y va al final. Ajusta `SECONDS_PER_MODEL` con el tiempo real.

In [ ]:
import math
import protocol as P
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 32, 64, 128]
M = max(LADDER)
PLAN = [
    dict(tag='A_k2_spectral', latent=2, beta_factor=FACTOR, seeds=list(range(8)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral'),
    dict(tag='B_k2_shuffled', latent=2, beta_factor=FACTOR, seeds=list(range(8)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='shuffled', mode_order_seed=12345),
    dict(tag='C_direct', latent=2, beta_factor=FACTOR, seeds=list(range(4)), modes=LADDER,
         models=['direct'], m_protocols=['scratch']),
    dict(tag='D_plainvae_k2', latent=2, beta_factor=FACTOR, seeds=list(range(8)), modes=[LADDER[0]],
         models=['plainvae'], m_protocols=['scratch']),
    dict(tag='E_k8_spectral', latent=8, beta_factor=FACTOR, seeds=list(range(4)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral'),
]
SECONDS_PER_MODEL = 80
print('pasos de Adam por modelo =', max(COMMON['epochs'] * math.ceil(0.8 * N / COMMON['batch']), COMMON['min_steps']))
P.show_plan(PLAN, 'mnist', SECONDS_PER_MODEL, COMMON)

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

Todas las comparaciones usan el **test**; la elección de m para M4 usa **validación**. Las filas de inicio de cadena `warm` (= modelo desde cero en m = 2) se incluyen en las curvas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from IPython.display import display, Image
import protocol as P
R = {p['tag']: P.load(p['out']) for p in PLAN}
A, B = R['A_k2_spectral'], R['B_k2_shuffled']
fmt = {'mse_x_mediana': '{:.4g}', 'knn_label_acc_mediana': '{:.3f}'}

# ---- M1: información del prefijo ----
if A is not None:
    info = P.information_table(A, 'mse_x', 'affine_x').add_prefix('afín_x_')
    lab = A[A.model == 'affine_x'].groupby('m').knn_label_accuracy.median().rename('kNN_dígito_prefijo')
    info = info.join(lab)
    if R['C_direct'] is not None:
        info = info.join(P.trained(R['C_direct']).groupby('m').mse_x.median().rename('direct_mse_x'))
    display(info.style.format('{:.4g}').set_caption('M1 · información del prefijo (mediana sobre réplicas)'))

# ---- tablas de reconstrucción ----
tablas = {tag: P.recon_table(d, tag) for tag, d in R.items() if d is not None}
for tag, t in tablas.items():
    display(t.style.format(fmt).set_caption(tag))

# ---- M2: continuación frente a desde cero (pareado por réplica) ----
if A is not None:
    pares = pd.DataFrame([r for m in LADDER[1:] if (r := P.paired_warm_vs_scratch(A, m))])
    display(pares.style.format({'scratch_mediana': '{:.4g}', 'warm_mediana': '{:.4g}', 'wilcoxon_p': '{:.3g}'})
            .set_caption('M2 · MSE_x test, pareado por réplica (warm_mejor = réplicas donde warm < scratch)'))
    rM = pares[pares.m == M]
    if len(rM):
        print(f"Criterio M2 (Wilcoxon en m = {M}, p < 0.05): "
              f"{'CUMPLIDO' if rM.wilcoxon_p.iloc[0] < .05 else 'NO cumplido'} (p = {rM.wilcoxon_p.iloc[0]:.3g})")

# ---- M3: orden espectral frente a aleatorio en m = M ----
if A is not None and B is not None:
    res = P.order_test(A, B, M, unfolding=False)
    display(res.style.format({'mannwhitney_p': '{:.3g}', 'mse_espectral': '{:.4g}', 'mse_aleatorio': '{:.4g}'})
            .set_caption(f'M3 · m = {M}: mismos modos, misma clase, distinto camino'))
    for _, r in res.iterrows():
        print(f"{r.model:9s} {'control' if r.model == 'vae' else 'prueba'}: Mann-Whitney p = {r.mannwhitney_p:.3g}")

# ---- M4: comparador plainvae ----
if A is not None and R['D_plainvae_k2'] is not None:
    w = P.trained(A)[lambda t: t.model == 'vae_warm']
    m_sel = int(w.groupby('m').validation_mse_x.mean().idxmin())
    a = w[w.m == m_sel].mse_x
    b = P.trained(R['D_plainvae_k2']).mse_x
    p = mannwhitneyu(a, b, alternative='less').pvalue
    print(f"M4: vae_warm (m = {m_sel}, elegido por validación) MSE_x mediana {a.median():.4g} | "
          f"plainvae {b.median():.4g} | cociente {a.median()/b.median():.3g} | Mann-Whitney (warm < plainvae) p = {p:.3g}")

In [ ]:
# ---- Figura resumen (1x3) ----
fig, (a, b, c) = plt.subplots(1, 3, figsize=(17, 4.6))
if A is not None:
    q = A[A.model == 'affine_x'].groupby('m').mse_x
    a.plot(q.median().index, q.median().values, 's--', color='gray', label='lectura afín de x (sin entrenar)')
if R['C_direct'] is not None:
    q = P.trained(R['C_direct']).groupby('m').mse_x
    a.plot(q.median().index, q.median().values, 'o-', color='C2', label='decoder direct (sin cuello de botella)')
    a.fill_between(q.median().index, q.quantile(.25).values, q.quantile(.75).values, color='C2', alpha=.2, lw=0)
a.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$', title='(a) Información del prefijo sobre x')
if A is not None:
    a2 = a.twinx()
    lab = A[A.model == 'affine_x'].groupby('m').knn_label_accuracy.median()
    a2.plot(lab.index, lab.values, 'd:', color='C9', label='kNN dígito del prefijo (sonda)')
    a2.set(ylabel='exactitud kNN del dígito', ylim=(0, 1))
    a2.legend(fontsize=7, loc='upper center')
a.legend(fontsize=7, loc='lower left')

estilos = [('A_k2_spectral', 'vae_warm', 'C4', '-', 'o', 'latente 2, continuación'),
           ('A_k2_spectral', 'vae', 'C0', ':', '^', 'latente 2, desde cero'),
           ('B_k2_shuffled', 'vae_warm', 'C1', '--', 's', 'latente 2, continuación, orden aleatorio'),
           ('B_k2_shuffled', 'vae', 'C7', ':', 'v', 'latente 2, desde cero, orden aleatorio'),
           ('E_k8_spectral', 'vae_warm', 'C5', '-', 'P', 'latente 8, continuación')]
for tag, model, col, ls, mk, lab in estilos:
    if R.get(tag) is None:
        continue
    g = P.with_chain_start(R[tag])
    g = g[g.model == model]
    s = g.groupby('m').mse_x
    b.plot(s.median().index, s.median().values, color=col, ls=ls, marker=mk, label=lab)
    b.fill_between(s.median().index, s.quantile(.25).values, s.quantile(.75).values, color=col, alpha=.12, lw=0)
    if 'knn_label_accuracy' in g:
        k = g.groupby('m').knn_label_accuracy.median()
        c.plot(k.index, k.values, color=col, ls=ls, marker=mk, label=lab)
if R['D_plainvae_k2'] is not None:
    d = P.trained(R['D_plainvae_k2'])
    b.axhline(d.mse_x.median(), color='C3', ls='-.', lw=1, label='plainvae latente 2')
    c.axhline(d.knn_label_accuracy.median(), color='C3', ls='-.', lw=1, label='plainvae latente 2')
b.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (mediana, IQR)', title='(b) Reconstrucción')
b.legend(fontsize=7)
c.set(xscale='log', ylim=(0, 1), xlabel='m', ylabel='exactitud kNN del dígito desde el latente',
      title='(c) Sonda de estructura de clases (no es una carta)')
c.legend(fontsize=7)
for axis in (a, b, c):
    axis.set_xticks(LADDER); axis.set_xticklabels([str(m) for m in LADDER], fontsize=8); axis.minorticks_off()
fig.tight_layout()
fig.savefig('/kaggle/working/mnist_protocolo.png', dpi=200); fig.savefig('/kaggle/working/mnist_protocolo.pdf')
plt.show()
if tablas:
    pd.concat(tablas.values()).to_csv('/kaggle/working/mnist_protocolo_tablas.csv', index=False)
f = Path(PLAN[0]['out']) / 'mnist_ensembles.png'
if f.exists():
    print('Referencia / D(mu) / E_q[D(z)] / muestras del prior, configuración elegida por validación:')
    display(Image(filename=str(f)))

## Qué descargar

`/kaggle/working/results_mnist_*`, `beta_sweep_mnist/`, `beta_confirm_mnist/`, `mnist_protocolo.png/.pdf` y `mnist_protocolo_tablas.csv`.